In [13]:
import numpy as np
import requests
import time
import logging
from typing import Optional
try:
    from xopt import Xopt
    from xopt.vocs import VOCS
    from xopt.evaluator import Evaluator
    from xopt.generators.bayesian import UpperConfidenceBoundGenerator
    XOPT_AVAILABLE = True
except ImportError:
    XOPT_AVAILABLE = False
logger = logging.getLogger(__name__)

# Setting up the API keys and trying a few submissions

In [14]:
API_KEY = "key_123"
BASE_URL = "https://halavanau.group/slacathon26"

headers = {
    "X-API-Key": API_KEY,
    "Content-Type": "application/json"
}

def run_validation(values, poll_interval=2):
    # Convert the 5-element list to the beamline task input dict
    # (q1, q2, q3, d2, d3) — this is what the active task expects
    input_dict = {
        "q1": values[0],
        "q2": values[1],
        "q3": values[2],
        "d2": values[3],
        "d3": values[4]
    }

    # Submit job using the current format: {"input": {...}}
    r = requests.post(f"{BASE_URL}/validate", headers=headers, json={"input": input_dict})
    r.raise_for_status()
    job = r.json()
    job_id = job["job_id"]
    print(f"Job started: {job_id} (status={job['status']})")

    # Poll until done
    while True:
        j = requests.get(f"{BASE_URL}/jobs/{job_id}", headers=headers).json()
        if j["status"] == "completed":
            return j.get("result")
        print(f"  ... still {j['status']}")
        time.sleep(poll_interval)


# Your two submissions
# Values are in order [q1, q2, q3, d2, d3] for the beamline task
result1 = run_validation([2.2547133301706257, -2.223405741870012, 0.9588998760031707, 0.033, 1.413])
print("First result:", result1)

result2 = run_validation([2.5537242710909087, -2.518264797355262, 1.0860652900429026, 0.033, 1.413])
print("Second result:", result2)

Job started: c7f83ea4-9f48-4b2a-ac16-04fc06ac6cba (status=processing)
  ... still processing
First result: {'score': 1.5889397682278592, 'solved': False, 'message': 'Objective is 1.5889397682278592, expected minimal (less than 1e-4)', 'evaltime': 0.0006225109100341797}
Job started: 582f8115-579b-47fb-83b5-e09416b210f9 (status=processing)
  ... still processing
Second result: {'score': 0.418888330879873, 'solved': False, 'message': 'Objective is 0.418888330879873, expected minimal (less than 1e-4)', 'evaltime': 0.0006558895111083984}


## Checking history of my submissions (10 last submissions)

In [15]:
response = requests.get(
    f"{BASE_URL}/history",
    headers={"X-API-Key": API_KEY}
)
if response.status_code == 200:
    data = response.json()
    logger.info(f"Total submissions: {data['total_submissions']}")
    logger.info(f"History: {data['history']}")

# Xopt optimizer implementation

In [16]:
class XoptOptimizer:
    """
    Lightweight client for running Xopt on the SLACATHON API.

    This follows the standard Xopt usage pattern:

        vocs = client.vocs
        evaluator = Evaluator(function=client.evaluate)
        generator = UpperConfidenceBoundGenerator(vocs=vocs)
        X = Xopt(evaluator=evaluator, generator=generator, vocs=vocs)

        X.random_evaluate(3)
        for _ in range(20):
            X.step()

        print(X.data)
        best = vocs.select_best(X.data)

    It also provides a convenient .optimize() method that returns
    results in a similar shape to GPOptimizer for easy comparison.
    """

    def __init__(self, api_key: str, base_url: str):
        self.api_key = api_key
        self.base_url = base_url.rstrip("/")

        self.session = requests.Session()
        self.session.headers.update({
            "X-API-Key": api_key,
            "Content-Type": "application/json"
        })

        self.task_info = self._request("GET", "/task")
        self.input_labels = self.task_info.get("parameter_labels") or ["x1", "x2", "x3", "x4", "x5"]
        self.bounds = self.task_info.get("bounds") or [(-10.0, 10.0)] * len(self.input_labels)

        logger.info(f"XoptOptimizer initialized for task: {self.task_info.get('name')}")

    def _request(self, method: str, path: str, **kwargs) -> dict:
        url = f"{self.base_url}{path}"
        try:
            resp = self.session.request(method, url, timeout=kwargs.pop("timeout", 30), **kwargs)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            logger.error(f"{method} {path} failed: {e}")
            return {}

    def _wait_for_job(self, job_id: str, timeout: float = 300.0, poll_interval: float = 1.5) -> dict:
        deadline = time.time() + timeout
        while time.time() < deadline:
            data = self._request("GET", f"/jobs/{job_id}")
            status = data.get("status")
            if status == "completed":
                return data.get("result", {})
            if status == "failed":
                logger.warning(f"Job {job_id} failed")
                return {}
            time.sleep(poll_interval)
        logger.warning(f"Job {job_id} timed out after {timeout}s")
        return {}

    @property
    def vocs(self):
        """Return a VOCS object built from the remote task definition."""
        if not XOPT_AVAILABLE:
            raise ImportError("xopt is required to create a VOCS object. pip install xopt")
        return VOCS(
            variables={label: [float(lo), float(hi)] for label, (lo, hi) in zip(self.input_labels, self.bounds)},
            objectives={"score": "MINIMIZE"},
        )

    def evaluate(self, inputs: dict) -> dict:
        """
        Xopt-compatible evaluation function.

        Takes a dict of the form {"q1": 1.2, "q2": -0.5, ...}
        Returns at least {"score": float}. Extra keys are stored by Xopt.
        """
        x = np.array([inputs[label] for label in self.input_labels])
        input_dict = {label: float(val) for label, val in zip(self.input_labels, x)}

        # Call the API
        job = self._request("POST", "/validate", json={"input": input_dict})
        job_id = job.get("job_id")
        if not job_id:
            return {"score": 1.0e10, "error": "failed to create job"}

        result = self._wait_for_job(job_id)

        score = result.get("score", 1.0e10) if result else 1.0e10
        if not np.isfinite(score):
            score = 1.0e10

        return {
            "score": float(score),
            "solved": result.get("solved", False) if result else False,
            "message": result.get("message", "") if result else "",
            "evaltime": result.get("evaltime", 0.0) if result else 0.0,
        }

    def query_objective(self, x: np.ndarray) -> tuple[float, dict]:
        """Convenience method compatible with GPOptimizer-style usage."""
        input_dict = {label: float(val) for label, val in zip(self.input_labels, x)}
        job = self._request("POST", "/validate", json={"input": input_dict})
        job_id = job.get("job_id")
        if not job_id:
            return float("inf"), {}
        result = self._wait_for_job(job_id)
        score = result.get("score", float("inf")) if result else float("inf")
        return score, (result or {})

    def submit_to_leaderboard(self, x: np.ndarray) -> dict:
        input_dict = {label: float(val) for label, val in zip(self.input_labels, x)}
        return self._request("POST", "/submit", json={"input": input_dict}, timeout=15)

    def submit_best_to_leaderboard(self, x: np.ndarray) -> dict:
        return self.submit_to_leaderboard(x)

    def view_leaderboard(self) -> dict:
        return self._request("GET", "/leaderboard")

    def get_history(self) -> Optional[dict]:
        return self._request("GET", "/history")

    def optimize(
        self,
        bounds: list[tuple] = None,
        n_iterations: int = 50,
        target_score: float = 1e-3,
        n_initial: int = 5,
        **xopt_options,
    ) -> tuple[Optional[np.ndarray], float, dict]:
        """
        High-level convenience method that follows the standard Xopt pattern
        under the hood and returns (best_x, best_score, result) for compatibility.
        """
        if not XOPT_AVAILABLE:
            raise ImportError("pip install xopt is required to use optimize()")

        if bounds is None:
            bounds = self.bounds

        vocs = self.vocs
        evaluator = Evaluator(function=self.evaluate)
        generator = UpperConfidenceBoundGenerator(vocs=vocs)

        X = Xopt(
            vocs=vocs,
            evaluator=evaluator,
            generator=generator,
            **xopt_options,
        )

        logger.info(f"Starting Xopt optimization, target < {target_score}")

        if n_initial > 0:
            X.random_evaluate(n_initial)

        for _ in range(n_iterations):
            X.step()
            if len(X.data) > 0:
                current_best = X.data["score"].min()
                if current_best < target_score:
                    logger.info(f"Target reached: {current_best:.6f}")
                    break

        df = X.data
        if len(df) == 0 or "score" not in df.columns:
            return None, float("inf"), {}

        best_idx = df["score"].idxmin()
        best_row = df.loc[best_idx]

        best_x = np.array([best_row[label] for label in self.input_labels])
        best_score = float(best_row["score"])

        best_result = {
            "score": best_score,
            "solved": bool(best_row.get("solved", False)),
            "message": best_row.get("message", ""),
            "evaltime": float(best_row.get("evaltime", 0.0)),
        }

        # Expose the full Xopt object for advanced use
        self.xopt = X

        logger.info(f"Done. Best score={best_score:.6f}")
        return best_x, best_score, best_result

In [19]:
# ====================== CONFIGURATION ======================
API_KEY = "key_123"
BASE_URL = "https://halavanau.group/slacathon26"

# Fixed values for some parameters (we discover which ones are fixed automatically below)
FIXED_VALUES = [1.0, 1.4]

# Free variables we want to optimize over
FREE_LABELS = ["q1", "q2", "q3"]
FREE_BOUNDS = [(1.0, 3.0), (-3.0, -2.0), (0.0, 2.0)]

client = XoptOptimizer(api_key=API_KEY, base_url=BASE_URL)

print("Task input labels from server:", client.input_labels)

# Automatically figure out which labels are fixed (the ones not in FREE_LABELS)
fixed_labels = [lbl for lbl in client.input_labels if lbl not in FREE_LABELS]
assert len(fixed_labels) == len(FIXED_VALUES), "Number of fixed values must match number of fixed labels"

# Create a VOCS using only the free variables
vocs = VOCS(
    variables={label: bnd for label, bnd in zip(FREE_LABELS, FREE_BOUNDS)},
    objectives={"score": "MINIMIZE"},
)

def evaluate_fixed(inputs: dict) -> dict:
    """Inject the fixed parameters (using labels discovered from the task), then evaluate."""
    full_inputs = dict(inputs)  # copy the free variables

    # Inject fixed values
    for label, value in zip(fixed_labels, FIXED_VALUES):
        full_inputs[label] = value

    # Build the full array in the exact order the task expects
    full_x = np.array([full_inputs[label] for label in client.input_labels])

    score, result = client.query_objective(full_x)

    return {
        "score": score if np.isfinite(score) else 1e10,
        "solved": result.get("solved", False),
        "message": result.get("message", ""),
        "evaltime": result.get("evaltime", 0.0),
    }

evaluator = Evaluator(function=evaluate_fixed)
generator = UpperConfidenceBoundGenerator(vocs=vocs)

X = Xopt(evaluator=evaluator, generator=generator, vocs=vocs)

print("Running Xopt optimization with fixed parameters...")
X.random_evaluate(5)
for _ in range(30):
    X.step()

print("\nOptimization results:")
print(X.data.tail())

# Get best from the DataFrame directly (more reliable than select_best which returns a tuple)
if len(X.data) > 0 and 'score' in X.data.columns:
    best_idx = X.data['score'].idxmin()
    best_row = X.data.loc[best_idx]

    best_x = np.array([best_row[label] for label in FREE_LABELS])
    full_input_dict = {label: val for label, val in zip(FREE_LABELS, best_x)}
    for label, val in zip(fixed_labels, FIXED_VALUES):
        full_input_dict[label] = val
    full_x = np.array([full_input_dict[label] for label in client.input_labels])

    print(f"\nBest (free): {best_x}")
    print(f"Full input:  {full_x}")
    print(f"Best score:  {best_row['score']:.6f}")

    sub = client.submit_to_leaderboard(full_x)
    if sub:
        print(f"Submitted! Rank: {sub.get('rank')}/{sub.get('leaderboard_size')}")
else:
    print("No valid result found.")

Task input labels from server: ['q1', 'q2', 'q3', 'd2', 'd3']
Running Xopt optimization with fixed parameters...

Optimization results:
     q1   q2        q3        score  solved  \
30  3.0 -2.0  1.556084   517.799269   False   
31  1.0 -2.0  1.019908    95.882758   False   
32  1.0 -2.0  1.925357   174.755167   False   
33  1.0 -3.0  1.030744   366.725682   False   
34  3.0 -3.0  1.349173  1610.338973   False   

                                              message  evaltime  xopt_runtime  \
30  Objective is 517.7992689961264, expected minim...  0.000674      1.756378   
31  Objective is 95.88275780845986, expected minim...  0.000731      1.756317   
32  Objective is 174.75516689953506, expected mini...  0.000428      1.758202   
33  Objective is 366.72568242577773, expected mini...  0.000595      1.763402   
34  Objective is 1610.338972897725, expected minim...  0.000731      1.756104   

    xopt_error  
30       False  
31       False  
32       False  
33       False  
34       

## View leaderboard

In [21]:
print("\n" + "="*60)
print("CURRENT LEADERBOARD")
print("="*60)
response = client.view_leaderboard()
if response:
    for i, entry in enumerate(response.get('leaderboard', []), 1):
        solved_marker = "✓" if entry.get('solved', False) else "✗"
        score = entry.get('score', 0)
        user = entry.get('user', 'Unknown')
        print(f"{i:2d}. [{solved_marker}] Score: {score:8.6f} | User: {user}")
    print(f"\nTotal entries: {response.get('total_entries', 'N/A')}")


CURRENT LEADERBOARD
 1. [✗] Score: 0.262394 | User: Alex
 2. [✗] Score: 0.262394 | User: Alex
 3. [✗] Score: 0.801639 | User: Alex
 4. [✗] Score: 0.884091 | User: Alex
 5. [✗] Score: 0.949626 | User: Alex
 6. [✗] Score: 1.790533 | User: Alex
 7. [✗] Score: 26.547862 | User: Alex
 8. [✗] Score: 26.799324 | User: Alex
 9. [✗] Score: 38.319862 | User: Alex
10. [✗] Score: 53.755836 | User: Alex
11. [✗] Score: 59.479829 | User: Alex
12. [✗] Score: 63.926096 | User: Alex
13. [✗] Score: 66.813550 | User: Alex
14. [✗] Score: 69.531951 | User: Alex
15. [✗] Score: 72.750735 | User: Alex

Total entries: 15
